In [ ]:
import pandas as pd
from shapely.geometry import LineString
from shapely.wkt import dumps
import numpy as np
from scipy.spatial import cKDTree

def transfer_connector_builder(
    base_node_df: pd.DataFrame,
    merge_node_df: pd.DataFrame,
    search_radius: float = 1000, # Default search radius in meters
    filter_node_type: str = None
) -> pd.DataFrame:
    """
    Builds connector links between the closest nodes of a base network and a merge network.
    It creates bidirectional links and includes various link attributes.

    Args:
        base_node_df (pd.DataFrame): DataFrame of nodes for the base network.
                                     Must contain 'node_id', 'x_coord', 'y_coord' columns.
        merge_node_df (pd.DataFrame): DataFrame of nodes for the network to be merged.
                                      Must contain 'node_id', 'x_coord', 'y_coord' columns.
        search_radius (float): The maximum distance (in meters) to search for a
                               closest base node for each merge node. Defaults to 1000 meters.
        filter_node_type (str, optional): The type of node to filter in merge_node_df (e.g., 'bus_service_node'). 
                                          If None, no filtering is applied. Defaults to None.

    Returns:
        pd.DataFrame: A DataFrame containing the generated connector links with all specified attributes.
                      Returns an empty DataFrame if no connectors are found or input DataFrames are invalid.
    """

    # Validate input DataFrames
    required_cols = ['node_id', 'x_coord', 'y_coord']
    if not all(col in base_node_df.columns for col in required_cols):
        print(f"Error: base_node_df is missing required columns. Ensure it has {required_cols}")
        return pd.DataFrame()
    if not all(col in merge_node_df.columns for col in required_cols):
        print(f"Error: merge_node_df is missing required columns. Ensure it has {required_cols}")
        return pd.DataFrame()
    
    # Select node_type to filter nodes for connector building
    if filter_node_type is not None:
        if 'node_type' in merge_node_df.columns:
            merge_node_df = merge_node_df[merge_node_df['node_type'] == filter_node_type]
            merge_node_df = merge_node_df.reset_index(drop=True)
        else:
            print(f"Warning: 'node_type' column is missing in merge_node_df. Skipping filter.")

    # Prepare coordinates for spatial search
    # Assuming x_coord is longitude and y_coord is latitude for haversine distance
    base_coords = base_node_df[['x_coord', 'y_coord']].to_numpy()
    merge_coords = merge_node_df[['x_coord', 'y_coord']].to_numpy()

    # Use cKDTree for efficient nearest neighbor search
    # This will find the closest base node for each merge node
    tree = cKDTree(base_coords)
    # query returns: distances, indices
    # distances are in the same unit as the coordinates, which are degrees
    # We divide by 111320.0 (approx meters per degree at equator) to convert search_radius to degrees.
    # This is a rough approximation for initial candidate filtering; for more accurate distance,
    # Haversine formula is needed *after* finding candidates.
    distances_deg, base_node_indices = tree.query(merge_coords, k=1, distance_upper_bound=search_radius / 111320.0)
    
    connector_links_data = []
    current_link_id = 1
    R = 6371000 # Earth radius in meters

    for i, (merge_node_id, merge_x, merge_y) in merge_node_df[['node_id', 'x_coord', 'y_coord']].iterrows():
        # Check if a candidate base node was found within the approximate radius
        if base_node_indices[i] < len(base_node_df): # Ensure index is valid (not infinity from distance_upper_bound)
            closest_base_node_idx = base_node_indices[i]
            base_node_id = base_node_df.iloc[closest_base_node_idx]['node_id']
            base_x = base_node_df.iloc[closest_base_node_idx]['x_coord']
            base_y = base_node_df.iloc[closest_base_node_idx]['y_coord']

            # Calculate precise Haversine distance in meters
            # Convert degrees to radians
            lat1, lon1, lat2, lon2 = map(np.radians, [merge_y, merge_x, base_y, base_x])
            dlon = lon2 - lon1
            dlat = lat2 - lat1
            a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
            c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
            length_meters = R * c

            # Filter by actual search radius in meters
            if length_meters <= search_radius:
                # Create bidirectional connector links

                # Connector: Merge Node -> Base Node
                geometry_wkt_1 = dumps(LineString([(merge_x, merge_y), (base_x, base_y)]))
                connector_links_data.append({
                    "link_id": current_link_id,
                    "name": "",
                    "from_node_id": merge_node_id,
                    "to_node_id": base_node_id,
                    "directed": 1,
                    "geometry": geometry_wkt_1,
                    "dir_flag": 1,
                    "length": length_meters,
                    "lanes": 1,
                    "free_speed": 120,
                    "capacity": 100000,
                    "link_type_name": "connector",
                    "link_type": 0,
                    "allowed_uses": "bike",
                    "from_biway": 1,
                    "is_link": 0,
                    "vdf_toll": 0,
                    "vdf_alpha": 0.15,
                    "vdf_beta": 4,
                    "vdf_plf": 1,
                    "vdf_length_mi": length_meters / 1609.344,
                    "vdf_free_speed_mph": 120/1.609344
                })
                current_link_id += 1

                # Connector: Base Node -> Merge Node (reverse direction)
                geometry_wkt_2 = dumps(LineString([(base_x, base_y), (merge_x, merge_y)]))
                connector_links_data.append({
                    "link_id": current_link_id,
                    "name": "",
                    "from_node_id": base_node_id,
                    "to_node_id": merge_node_id,
                    "directed": 1,
                    "geometry": geometry_wkt_2,
                    "dir_flag": 1,
                    "length": length_meters,
                    "lanes": 1,
                    "free_speed": 120,
                    "capacity": 100000,
                    "link_type_name": "connector",
                    "link_type": 0,
                    "allowed_uses": "bike",
                    "from_biway": 1,
                    "is_link": 0,
                    "vdf_toll": 0,
                    "vdf_alpha": 0.15,
                    "vdf_beta": 4,
                    "vdf_plf": 1,
                    "vdf_length_mi": length_meters / 1609.344,
                    "vdf_free_speed_mph": 120/1.609344
                })
                current_link_id += 1
    
    if not connector_links_data:
        print("No connector links found within the specified search radius.")
        return pd.DataFrame()

    connector_df = pd.DataFrame(connector_links_data)

    # Calculate 'vdf_free_speed_in_mph'
    connector_df['vdf_free_speed_in_mph'] = connector_df['free_speed'] / 1.609344

    # Calculate 'vdf_fftt'
    # Note: free_speed can be 0, which would result in division by zero (infinity)
    connector_df['vdf_fftt'] = (connector_df['length'] / 1000) / connector_df['free_speed'] * 60

    # Handle potential infinite values resulting from division by zero
    # We replace infinity (inf) with 0, a common practice for impassable links
    connector_df.replace([np.inf, -np.inf], 0, inplace=True)


    return connector_df

In [ ]:
#import os
#os.chdir("file path")

updated_base_node_df = pd.read_csv('updated_nodes.csv')
updated_merge_node_df = pd.read_csv('updated_bikeway_nodes.csv')

connector_df = transfer_connector_builder(
    base_node_df=updated_base_node_df,
    merge_node_df=updated_merge_node_df,
    search_radius=200,
    filter_node_type=None
)

connector_df.to_csv('transfer_connector_links.csv', index=False)